# Python bridge for 03a

## A short code refresher

Use this optional notebook alongside 03a if you would like a closer look at the Python mechanics behind scaling and pairwise distance calculations. It takes about 15–20 minutes and uses four records that can be inspected directly.

The 03a lecture develops the geometry and its interpretation. This companion focuses on five code patterns:

1. standardize DataFrame columns;
2. preserve a two-dimensional row for SciPy;
3. calculate distances with `cdist`;
4. exclude self-distances; and
5. map nearest positions back to record names.

## Create a feature table

Rows represent records and columns represent numerical features. This rows-by-features shape is the representation expected by the distance code.

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

In [ ]:
features = pd.DataFrame(
    {
        "feature_1": [10, 13, 12, 11],
        "feature_2": [50, 50, 52, 51],
        "feature_3": [2, 2, 2, 6],
    },
    index=["A", "B", "C", "D"],
)

print("table shape:", features.shape)
features

The shape `(4, 3)` means four records by three features. The row labels identify records but are not numerical inputs to the distance calculation.

## Standardize the columns

`features.mean()` and `features.std()` return one value per column. The chained `.sub(...).div(...)` operation aligns those values with columns, subtracts each column's mean, and divides by its sample standard deviation.

In [ ]:
feature_means = features.mean()
feature_sds = features.std(ddof=1)

standardized = features.sub(feature_means).div(feature_sds)

standardized.round(3)

Check the result using the same column-wise operations. Small values near zero can appear because computers store decimal values with finite precision.

In [ ]:
pd.DataFrame(
    {
        "standardized mean": standardized.mean(),
        "standardized sample SD": standardized.std(ddof=1),
    }
).round(10)

`ddof=1` requests the sample standard deviation used in the primary notebook. Using the same setting in the transformation and the check keeps the two calculations consistent.

## Preserve a two-dimensional row

Single brackets and double brackets produce different objects with different shapes. `.loc["A"]` returns a one-dimensional `Series`; `.loc[["A"]]` returns a one-row, two-dimensional `DataFrame`.

In [ ]:
row_series = standardized.loc["A"]
one_row_frame = standardized.loc[["A"]]

print("Series shape:", row_series.shape)
print("one-row DataFrame shape:", one_row_frame.shape)

`cdist` expects each input to have shape `(number of records, number of features)`. Double brackets preserve the `(1, 3)` shape needed to represent one record with three features.

## Inspect coordinate gaps

The expression below aligns the two rows by column name, subtracts their values, and takes the absolute value of every coordinate gap.

In [ ]:
absolute_gaps = features.loc["A"].sub(features.loc["C"]).abs()

absolute_gaps

The methods form a mechanical chain: select two rows, subtract corresponding coordinates, then remove the signs. The primary notebook explains how a distance metric combines these gaps.

## Calculate one pairwise distance

`cdist` compares every row in its first input with every row in its second input. Comparing one row with one row therefore returns an array with shape `(1, 1)`.

In [ ]:
point_a = standardized.loc[["A"]]
point_c = standardized.loc[["C"]]

pair_distance = cdist(point_a, point_c, metric="euclidean")

print("returned array:", pair_distance)
print("returned shape:", pair_distance.shape)
print("single distance:", pair_distance.item())

`.item()` extracts the single number from the one-by-one array. Keep the array when several pairwise distances are needed.

## Build a complete distance matrix

Passing the complete feature table twice compares every record with every record. `.to_numpy()` removes the row and column labels before SciPy performs the numerical calculation.

In [ ]:
distance_array = cdist(
    standardized.to_numpy(),
    standardized.to_numpy(),
    metric="euclidean",
)

distance_frame = pd.DataFrame(
    distance_array,
    index=standardized.index,
    columns=standardized.index,
)

distance_frame.round(3)

The output has shape `(4, 4)`. Row `A`, column `C` is the distance from `A` to `C`. The diagonal contains zeros because each record is identical to itself.

## Locate each nearest other record

The zero diagonal would make every record its own nearest match. `np.fill_diagonal(distance_array, np.inf)` replaces those zeros with infinity so they cannot be selected as minima.

In [ ]:
np.fill_diagonal(distance_array, np.inf)

nearest_positions = distance_array.argmin(axis=1)
record_names = standardized.index.to_numpy()
nearest_names = record_names[nearest_positions]

nearest_peers = pd.Series(
    nearest_names,
    index=standardized.index,
    name="nearest recorded peer",
)

nearest_peers

`argmin(axis=1)` returns the column position of the smallest value in each row. Indexing `record_names` with those positions converts numerical positions such as `0` and `2` back to labels such as `A` and `C`.

The modified array now contains infinity on its diagonal. Rebuild `distance_frame` if you want the displayed table to reflect that later change.

## Inspect one anchor's ranking

A labeled distance frame makes one record's candidates easy to inspect. `.sort_values()` orders them from smallest distance to largest, and `.head(3)` keeps the first three.

In [ ]:
distance_frame_with_exclusions = pd.DataFrame(
    distance_array,
    index=standardized.index,
    columns=standardized.index,
)

distance_frame_with_exclusions.loc["A"].sort_values().head(3)

The nearest record appears first. The excluded self-distance does not appear because infinity sorts after every finite distance.

## Try one small modification

Change `metric="euclidean"` to `metric="cityblock"` in the complete-matrix cell, then rerun the nearest-record cells. The array shapes and indexing mechanics remain the same; the calculated distances and possibly the selected peers change.

## Ready for 03a

You are ready to return to 03a when you can recognize these patterns:

```text
frame.sub(means).div(sds)                  # standardize aligned columns
frame.loc[["A"]]                          # keep one row two-dimensional
cdist(first_rows, second_rows, metric=...) # calculate every requested pair
np.fill_diagonal(distances, np.inf)        # exclude zero self-distances
distances.argmin(axis=1)                   # find one minimum position per row
names[nearest_positions]                   # map positions back to record names
```

The important implementation checks are that rows represent records, columns appear in the same order for every comparison, `cdist` receives two-dimensional inputs, self-distances are excluded before selecting minima, and numerical positions are mapped back to the correct labels.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).